# Semenov et al. (2025), Figure S1, from dandiset 001272 (data elsewhere)

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025), *Cell Reports Physical Science* 6,
102660, [doi:10.1016/j.xcrp.2025.102660](https://doi.org/10.1016/j.xcrp.2025.102660); the claims
file was written from the NIH author manuscript (PMC12320893) with its supplement.

Dataset. The claim figs1-di8-anepps-control has `data: elsewhere`, in
[DANDI:001272](https://dandiarchive.org/dandiset/001272/0.250805.1358), version 0.250805.1358
("The Use of Di8-ANEPPS to Evaluate Neuronal Sensitivity to Unmodulated Sine Waves"), the most
recent published version; the paper lists the dandiset without a version.

| Claim | Tested here |
|---|---|
| figs1-di8-anepps-control | yes |

No random numbers are used.

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import fastimg

pd.set_option("display.width", 200)
started = time.time()
sessions = fastimg.read_all(fastimg.list_assets(fastimg.S1_DANDISET, fastimg.S1_VERSION))
print(f"{len(sessions)} sessions read from dandiset {fastimg.S1_DANDISET} version {fastimg.S1_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ("subject_id", "amplitude_v", "species", "rate_hz", "shape")} for s in sessions])
print("Sessions per subject id:", dict(Counter(info.subject_id)))
print("Species:", dict(Counter(info.species)), "; rates:", dict(Counter(info.rate_hz)), "; shapes:", dict(Counter(info["shape"])))
print("Amplitudes (V) per subject id:")
print(info.groupby("subject_id").amplitude_v.agg(["min", "max", "count"]).to_string())
print("Descriptions naming Di-8-ANEPPS:", sum("Di-8-ANEPPS" in s["description"] for s in sessions), "of", len(sessions))
print("Descriptions naming tetrodotoxin as present:", sum("presence of" in s["description"] for s in sessions))
print("\nDescription of one session:\n", sessions[0]["description"])

The dandiset has three neurons (subject ids, all recorded on 2024-08-06) where the paper reports
eight. Each session is 1,200 images of 16 x 16 pixels, about 389 ms; the stimulus timing is not in
the files and is taken from the Methods (50 ms before the stimulus, 250 ms during).

The traces are inverted, as Di-8-ANEPPS emission falls with depolarization (Methods). The check
below compares the mean inverted dF before the stated onset, during the stimulus and after it,
over all sessions: a response confined to the stated window supports the timing.

In [ ]:
fs = info.rate_hz.iloc[0]
on, off = int(fastimg.S1_STIM_START_S * fs), int((fastimg.S1_STIM_START_S + fastimg.S1_STIM_S) * fs)
for s in sessions:
    s["dF"] = -fastimg.delta_f(s["frame_means"], s["rate_hz"], fastimg.S1_STIM_START_S)
    s["sd_pre"] = float(np.nanstd(s["dF"][fastimg.DROP_FRAMES:on]))
x = np.array([s["dF"] for s in sessions])
print(f"Mean inverted dF (%): before onset {np.nanmean(x[:, fastimg.DROP_FRAMES:on]):.2f}, "
      f"first 25 ms of the stimulus {np.nanmean(x[:, on:on + int(0.025 * fs)]):.2f}, "
      f"whole stimulus {np.nanmean(x[:, on:off]):.2f}, last 50 ms after it {np.nanmean(x[:, -int(0.05 * fs):]):.2f}")
step = np.nanmean(x[:, on:on + 10], axis=1) - np.nanmean(x[:, on - 10:on], axis=1)
print(f"Step at the stated onset (10 frames after minus 10 before), median over sessions: {np.median(step):.2f}%; "
      f"the same 50 frames earlier: {np.median(np.nanmean(x[:, on - 40:on - 30], 1) - np.nanmean(x[:, on - 50:on - 40], 1)):.2f}%")
print("Pre-stimulus SD of dF (%), range over sessions:", round(min(s["sd_pre"] for s in sessions), 2), "to",
      round(max(s["sd_pre"] for s in sessions), 2))

The inverted dF rises from the stated onset (the median step at the onset is larger than the same
step 50 frames earlier) but stays up after the stimulus, so part of the rise is the bleaching of
the dye, which the inversion turns into an apparent depolarization. The stated timing is consistent
with the data, not shown by them.

## figs1-di8-anepps-control

> With Di-8-ANEPPS instead of FluoVolt, in eight neurons, near-threshold stimulation elicited one or
> a few delayed APs and stronger fields induced sustained firing ending in sustained
> depolarization, as with FluoVolt.

Criterion: traces inverted; AP detection readings (a) a peak above 5% dF, (b) a rise of at least
5% within 5 ms, (c) a peak above 5 SD of the pre-stimulus trace. Reproduced when, under every
reading, every neuron fires, first-AP latency falls with field in more than half, and the highest
field ends in depolarization as in fig3c-sustained-firing-2s in more than half. Taken here (standard
choices, as the criterion does not fix them): latency falls when the Spearman correlation of field
and first-AP latency over a neuron's sessions with APs is negative; the highest field ends in
depolarization when its sessions have no APs in the last 100 ms of the stimulus and mean dF above
5% there (the window of the claims file's block test for 250-ms stimuli, fig2-modulated-locks-to-beats).

In [ ]:
rows = []
last = off - int(0.1 * fs)
for s in sessions:
    row = {"neuron": s["subject_id"], "amplitude_v": s["amplitude_v"], "sd_pre": round(s["sd_pre"], 2),
           "mean_dF_last100": float(np.nanmean(s["dF"][last:off]))}
    for r in "abc":
        p = fastimg.detect(s["dF"], r, s["rate_hz"], level=5 * s["sd_pre"])
        p = p[(p >= on) & (p < off)]
        row[f"{r}_aps"] = len(p)
        row[f"{r}_latency_ms"] = 1000 * (p[0] - on) / fs if len(p) else np.nan
        row[f"{r}_last100"] = int((p >= last).sum())
    rows.append(row)
table = pd.DataFrame(rows).sort_values(["neuron", "amplitude_v"])
print(table.round(2).to_string(index=False))

In [ ]:
rows = []
for r in "abc":
    for neuron, sub in table.groupby("neuron"):
        fired = sub[sub[f"{r}_aps"] > 0]
        rho = stats.spearmanr(fired.amplitude_v, fired[f"{r}_latency_ms"]).correlation if fired.amplitude_v.nunique() > 1 else np.nan
        top = sub[sub.amplitude_v == sub.amplitude_v.max()]
        rows.append({"reading": r, "neuron": neuron, "sessions_with_aps": len(fired), "of": len(sub),
                     "fires": len(fired) > 0, "latency_rho": rho, "latency_falls": bool(rho < 0),
                     "highest_field_v": top.amplitude_v.iloc[0], "highest_aps_last100": int(top[f"{r}_last100"].sum()),
                     "highest_mean_dF_last100": top.mean_dF_last100.mean(),
                     "ends_depolarized": bool((top[f"{r}_last100"] == 0).all() and top.mean_dF_last100.mean() > 5)})
per = pd.DataFrame(rows)
print(per.round(2).to_string(index=False))
summary = per.groupby("reading").agg(all_fire=("fires", "all"), latency_falls=("latency_falls", "sum"),
                                     ends_depolarized=("ends_depolarized", "sum"), neurons=("neuron", "count"))
summary["holds"] = summary.all_fire & (summary.latency_falls > summary.neurons / 2) & (summary.ends_depolarized > summary.neurons / 2)
print("\n", summary.to_string())
print("\nLargest mean inverted dF over the last 100 ms of any session:", round(table.mean_dF_last100.max(), 2), "%")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
t = (np.arange(len(sessions[0]["dF"])) - on) / fs * 1000
lo, hi = np.inf, -np.inf
for ax, (neuron, sub) in zip(axes, sorted(((n, [s for s in sessions if s["subject_id"] == n]) for n in info.subject_id.unique()))):
    for s in sorted(sub, key=lambda s: s["amplitude_v"])[::max(1, len(sub) // 4)]:
        ax.plot(t[:len(s["dF"])], s["dF"], lw=0.5, label=f"{s['amplitude_v']} V")
        lo, hi = min(lo, np.nanmin(s["dF"])), max(hi, np.nanmax(s["dF"]))
    ax.axvline(0, c="k", ls=":"); ax.axvline(250, c="k", ls=":"); ax.set_ylim(-4, 8)
    ax.set_ylabel(neuron, fontsize=8); ax.legend(fontsize=6, ncol=4)
axes[-1].set_xlabel("time from the stated stimulus onset (ms)")
plt.show()
print(f"Inverted dF of the plotted sessions ranges {lo:.1f} to {hi:.1f}% (axes -4 to 8%)")

**Result: not reproduced.** The criterion holds under none of the three readings (`holds` is
False for a, b and c). In no neuron does the highest field end in depolarization by the
criterion's 5% level; the largest mean inverted dF over the last 100 ms of any session is 3.2%.
Every neuron fires under readings (a) and (c), and neuron 4_20240806 has no AP under reading (b).
First-AP latency falls with field in 2, 1 and 3 of the 3 neurons under readings (a), (b) and (c).
Reading (c), 5 SD of the pre-stimulus trace, finds APs in nearly every session, some within a few
ms of the onset. The Di-8-ANEPPS signals are smaller than the FluoVolt ones the 5% level comes
from, and the dataset has three neurons where the paper has eight. The failure may therefore lie in
applying the FluoVolt level to this dye.

## Summary

| Claim | Outcome |
|---|---|
| figs1-di8-anepps-control | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: F is the whole 16 x 16 frame; the stimulus timing is the
Methods' (not in the files); the readings of "latency falls" and "ends in depolarization" are
standard choices stated above.

What was found about the dataset: three neurons where the paper reports eight; no stimulus timing
in the files.